# Generate a Satellite Image from Geospatial Conditions

This notebook shows how to use our trained model to generate a **256 × 256 satellite-style image** from geospatial layers and a text prompt.

**Model:** Stable Diffusion v1.5 + LoRA + two ControlNet branches combined with learned fusion weights. All weights are public on Hugging Face:

| Component | Hugging Face repo |
|---|---|
| LoRA (satellite style) | [`safadakdouk123/sd-lora-satellite-mydata`](https://huggingface.co/safadakdouk123/sd-lora-satellite-mydata) |
| ControlNets + fusion | [`safadakdouk123/sd-controlnet-lebanon-fulldataset`](https://huggingface.co/safadakdouk123/sd-controlnet-lebanon-fulldataset) |
| Base model | [`runwayml/stable-diffusion-v1-5`](https://huggingface.co/runwayml/stable-diffusion-v1-5) |

No Hugging Face account or token is needed. The weights download automatically.

## Inputs

Six **single-band GeoTIFF** files, one per layer, plus a text prompt:

| Layer | Meaning | Expected values |
|---|---|---|
| DEM | Elevation | metres |
| Slope | Terrain slope | degrees (0–90) |
| Aspect | Slope direction | degrees (0–360), `-1` for flat |
| LULC | Land use / land cover | integer class codes (0–8) |
| Roads | Road mask | binary (0 / 1) |
| Rivers | River mask | binary (0 / 1) |

**Requirements:** the six rasters must be **aligned** (same size, CRS and transform) and be **256 × 256** tiles. This notebook does not create the layers from raw data.

**Hardware:** a GPU is recommended (about 10–20 s per image on a modern GPU). It also runs on CPU, but slowly.

In [ ]:
%pip install -q diffusers transformers accelerate huggingface_hub safetensors rasterio scipy pillow matplotlib

## 1. Configuration

Edit only this cell.

In [ ]:
from pathlib import Path

# ---------------- Input layers (one single-band GeoTIFF each) ----------------
DEM_PATH    = "inputs/dem.tif"
SLOPE_PATH  = "inputs/slope.tif"
ASPECT_PATH = "inputs/aspect.tif"
LULC_PATH   = "inputs/lulc.tif"
ROADS_PATH  = "inputs/roads.tif"
RIVERS_PATH = "inputs/rivers.tif"

# ---------------- Text prompt ----------------
PROMPT = "Summer Hilly Urban area in south Lebanon"   # <- describe the scene you want

# ---------------- Generation settings ----------------
NUM_INFERENCE_STEPS = 50     # more steps = slower, usually slightly cleaner
GUIDANCE_SCALE      = 1.0    # 1.0 = no classifier-free guidance (used in our evaluation)
SEED                = 42     # change the seed to get a different variation

# ---------------- Output ----------------
OUTPUT_DIR = Path("./generated_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------- Model repositories (public) ----------------
BASE_MODEL         = "runwayml/stable-diffusion-v1-5"
LORA_REPO_ID       = "safadakdouk123/sd-lora-satellite-mydata"
LORA_FILENAME      = "best/lora_weights.pt"
CONTROLNET_REPO_ID = "safadakdouk123/sd-controlnet-lebanon-fulldataset"
CONTROLNET_A_FILE  = "branch_A/best.pt"        # terrain: DEM, Slope, Aspect
CONTROLNET_B_FILE  = "branch_B/best.pt"        # semantic: LULC, Roads, Rivers
FUSION_FILE        = "fusion/best_fusion.pt"

LORA_ALPHA = 16   # training values; do not change
TILE_SIZE  = 256

In [ ]:
import numpy as np
import torch
import rasterio
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import binary_dilation, zoom
from huggingface_hub import hf_hub_download
from diffusers import (
    AutoencoderKL,
    UNet2DConditionModel,
    ControlNetModel,
    DDIMScheduler,
)
from transformers import CLIPTokenizer, CLIPTextModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DT = torch.float16 if DEVICE == "cuda" else torch.float32

print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found: generation will work but will be slow.")

## 2. Load the six conditioning layers

In [ ]:
LAYER_PATHS = {
    "DEM": DEM_PATH,
    "Slope": SLOPE_PATH,
    "Aspect": ASPECT_PATH,
    "LULC": LULC_PATH,
    "Roads": ROADS_PATH,
    "Rivers": RIVERS_PATH,
}

layers, metas = {}, {}
for name, path in LAYER_PATHS.items():
    if not Path(path).is_file():
        raise FileNotFoundError(f"{name}: file not found -> {path}")
    with rasterio.open(path) as src:
        arr = src.read(1).astype(np.float32)
        metas[name] = (src.height, src.width, src.crs, src.transform)
    arr[~np.isfinite(arr)] = 0.0
    layers[name] = arr

# All layers must be aligned
ref = metas["DEM"]
for name, m in metas.items():
    if m[:2] != ref[:2]:
        raise ValueError(f"{name} has size {m[:2]} but DEM has {ref[:2]}.")
    if m[2] != ref[2] or m[3] != ref[3]:
        print(f"Warning: {name} has a different CRS/transform from DEM. "
              "Make sure the layers are aligned.")

# The model was trained on 256 x 256 tiles
h, w = ref[:2]
if (h, w) != (TILE_SIZE, TILE_SIZE):
    print(f"Warning: input is {h}x{w}; resizing to {TILE_SIZE}x{TILE_SIZE}. "
          "For best results, provide 256x256 tiles.")
    for name in layers:
        order = 1 if name in ("DEM", "Slope", "Aspect") else 0
        layers[name] = zoom(layers[name], (TILE_SIZE / h, TILE_SIZE / w), order=order)

print({k: v.shape for k, v in layers.items()})

## 3. Preprocess the layers

These are the **exact normalizations used during training** (every layer is scaled to [-1, 1]). The layers are then grouped into the two ControlNet branches:

- **Branch A (terrain):** DEM, Slope, Aspect
- **Branch B (semantic):** LULC, Roads, Rivers

In [ ]:
def normalize_dem(dem):
    dem = dem.copy()
    nodata = (dem == -9999) | (dem == 32767) | (dem == -32768) | (~np.isfinite(dem))
    dem[nodata] = 0.0
    dem = np.clip(dem, 69.0, 2350.0)
    dem = np.clip((dem - 69.0) / (2350.0 - 69.0), 0, 1)
    dem[nodata] = 0.0
    return (dem * 2.0 - 1.0).astype(np.float32)

def normalize_slope(s):
    s = s.copy()
    nodata = (s < 0) | (~np.isfinite(s))
    s[nodata] = 0.0
    s = np.clip(s / 90.0, 0, 1)
    s[nodata] = 0.0
    return (s * 2.0 - 1.0).astype(np.float32)

def normalize_aspect(a):
    a = a.copy()
    nodata = (a == -1) | (~np.isfinite(a))
    a[nodata] = 0.0
    a = np.clip(a / 360.0, 0, 1)
    a[nodata] = 0.5
    return (a * 2.0 - 1.0).astype(np.float32)

def normalize_lulc(l):
    l = l.copy()
    l[~np.isfinite(l)] = 0.0
    return (np.clip(l / 8.0, 0, 1) * 2.0 - 1.0).astype(np.float32)

def normalize_roads(r):
    r = r.copy()
    r[~np.isfinite(r)] = 0.0
    r = binary_dilation((r > 0.5), iterations=1).astype(np.float32)
    return (r * 2.0 - 1.0).astype(np.float32)

def normalize_rivers(r):
    r = r.copy()
    r[~np.isfinite(r)] = 0.0
    r = binary_dilation((r > 0.5), iterations=2).astype(np.float32)
    return (r * 2.0 - 1.0).astype(np.float32)

branch_A_np = np.stack([
    normalize_dem(layers["DEM"]),
    normalize_slope(layers["Slope"]),
    normalize_aspect(layers["Aspect"]),
])
branch_B_np = np.stack([
    normalize_lulc(layers["LULC"]),
    normalize_roads(layers["Roads"]),
    normalize_rivers(layers["Rivers"]),
])

print("Branch A:", branch_A_np.shape, "| Branch B:", branch_B_np.shape)

## 4. Preview the input layers

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8.5))
specs = [
    ("DEM", "terrain"), ("Slope", "YlOrRd"), ("Aspect", "hsv"),
    ("LULC", "tab20"), ("Roads", "gray_r"), ("Rivers", "Blues"),
]
for ax, (name, cmap) in zip(axes.flat, specs):
    im = ax.imshow(layers[name], cmap=cmap)
    ax.set_title(name); ax.axis("off")
    if name in ("DEM", "Slope", "Aspect", "LULC"):
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "input_layers.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Load the model

The first run downloads the weights (about 6 GB in total). Later runs use the local cache.

In [ ]:
print("Loading VAE...")
vae = AutoencoderKL.from_pretrained(BASE_MODEL, subfolder="vae", torch_dtype=DT).to(DEVICE).eval()

print("Loading tokenizer and text encoder...")
tokenizer = CLIPTokenizer.from_pretrained(BASE_MODEL, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(BASE_MODEL, subfolder="text_encoder", torch_dtype=DT).to(DEVICE).eval()

print("Loading UNet...")
unet = UNet2DConditionModel.from_pretrained(BASE_MODEL, subfolder="unet", torch_dtype=DT).to(DEVICE).eval()

scheduler = DDIMScheduler.from_pretrained(BASE_MODEL, subfolder="scheduler")

for m in (vae, text_encoder, unet):
    m.requires_grad_(False)
print("Stable Diffusion components loaded.")

### Merge the LoRA weights into the UNet

In [ ]:
def load_checkpoint(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except Exception:
        return torch.load(path, map_location="cpu")

lora_path = hf_hub_download(LORA_REPO_ID, LORA_FILENAME, repo_type="model")
sd = load_checkpoint(lora_path)

# unwrap nested checkpoints if needed
if isinstance(sd, dict) and not any("lora_A" in k for k in sd if isinstance(k, str)):
    for k, v in sd.items():
        if isinstance(v, dict) and any("lora_A" in kk for kk in v):
            sd = v
            break

A_SUF, B_SUF = ".lora_A.weight", ".lora_B.weight"
names = [k[:-len(A_SUF)] for k in sd if isinstance(k, str) and k.endswith(A_SUF)]

def resolve(model, name):
    parts = name.split(".")
    for i in range(len(parts)):   # strip leading prefixes such as 'module.'
        try:
            return model.get_submodule(".".join(parts[i:]))
        except AttributeError:
            continue
    return None

merged, skipped = 0, []
for n in names:
    A = sd[n + A_SUF].float()
    B = sd[n + B_SUF].float()
    scale = LORA_ALPHA / A.shape[0]
    mod = resolve(unet, n)
    if mod is None or not hasattr(mod, "weight") or mod.weight.ndim != 2:
        skipped.append(n)
        continue
    delta = (scale * (B @ A)).to(mod.weight.device)
    mod.weight.data = (mod.weight.data.float() + delta).to(mod.weight.dtype)
    merged += 1

print(f"LoRA layers merged: {merged} | skipped: {len(skipped)}")
assert merged > 0, "No LoRA layers were merged."

### Load the two ControlNets and the fusion weights

In [ ]:
def load_controlnet(filename):
    cn = ControlNetModel.from_unet(unet).to(dtype=torch.float32)   # same architecture as training
    path = hf_hub_download(CONTROLNET_REPO_ID, filename, repo_type="model")
    cn.load_state_dict(load_checkpoint(path))
    return cn.to(DEVICE).eval().requires_grad_(False)

print("Loading ControlNet A (terrain)...")
controlnet_A = load_controlnet(CONTROLNET_A_FILE)
print("Loading ControlNet B (semantic)...")
controlnet_B = load_controlnet(CONTROLNET_B_FILE)

fusion = load_checkpoint(hf_hub_download(CONTROLNET_REPO_ID, FUSION_FILE, repo_type="model"))
alpha_A, alpha_B = float(fusion["alpha_A"]), float(fusion["alpha_B"])
print(f"Learned fusion weights: alpha_A={alpha_A:.4f}, alpha_B={alpha_B:.4f}")

## 6. Generation function

In [ ]:
@torch.no_grad()
def encode_prompt(prompt):
    L = tokenizer.model_max_length
    ids = tokenizer([prompt], padding="max_length", max_length=L,
                    truncation=True, return_tensors="pt").input_ids.to(DEVICE)
    uids = tokenizer([""], padding="max_length", max_length=L,
                     truncation=True, return_tensors="pt").input_ids.to(DEVICE)
    return torch.cat([text_encoder(uids)[0], text_encoder(ids)[0]])   # [uncond, cond]


@torch.no_grad()
def generate(prompt, branch_a, branch_b, seed=42, guidance=1.0, steps=50):
    cA = torch.from_numpy(branch_a).unsqueeze(0).to(DEVICE, torch.float32)
    cB = torch.from_numpy(branch_b).unsqueeze(0).to(DEVICE, torch.float32)
    emb = encode_prompt(prompt)

    h, w = branch_a.shape[-2:]
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    scheduler.set_timesteps(steps, device=DEVICE)
    lat = torch.randn(1, 4, h // 8, w // 8, generator=g, device=DEVICE, dtype=DT)
    lat = lat * scheduler.init_noise_sigma

    for t in scheduler.timesteps:
        li = torch.cat([lat] * 2)
        with torch.autocast(device_type=DEVICE, enabled=(DEVICE == "cuda")):
            dA, mA = controlnet_A(li.float(), t, encoder_hidden_states=emb.float(),
                                  controlnet_cond=torch.cat([cA] * 2), return_dict=False)
            dB, mB = controlnet_B(li.float(), t, encoder_hidden_states=emb.float(),
                                  controlnet_cond=torch.cat([cB] * 2), return_dict=False)
            # learned fusion of the two branches
            down = [(alpha_A * a + alpha_B * b).to(DT) for a, b in zip(dA, dB)]
            mid = (alpha_A * mA + alpha_B * mB).to(DT)
            eps = unet(li, t, encoder_hidden_states=emb,
                       down_block_additional_residuals=down,
                       mid_block_additional_residual=mid, return_dict=False)[0]
        e_u, e_t = eps.chunk(2)
        eps = e_u + guidance * (e_t - e_u)
        lat = scheduler.step(eps, t, lat).prev_sample

    img = vae.decode((lat / vae.config.scaling_factor).to(DT)).sample
    return img.clamp(-1, 1).float()   # (1, 3, H, W) in [-1, 1]


def to_pil(x):
    arr = ((x[0].permute(1, 2, 0).cpu().numpy() + 1) / 2).clip(0, 1)
    return Image.fromarray((arr * 255).round().astype(np.uint8))

## 7. Generate the image

In [ ]:
out = generate(PROMPT, branch_A_np, branch_B_np,
               seed=SEED, guidance=GUIDANCE_SCALE, steps=NUM_INFERENCE_STEPS)
image = to_pil(out)

save_path = OUTPUT_DIR / "generated_image.png"
image.save(save_path)
print("Saved to:", save_path)

plt.figure(figsize=(7, 7))
plt.imshow(image)
plt.title("Generated satellite-style image")
plt.axis("off")
plt.show()

### (Optional) Try several seeds

The same conditions can give different variations of the scene. The terrain, roads and rivers stay consistent while textures change.

In [ ]:
SEEDS = [1, 2, 3, 4]

fig, axes = plt.subplots(1, len(SEEDS), figsize=(4 * len(SEEDS), 4.2))
for ax, sd in zip(axes, SEEDS):
    im = to_pil(generate(PROMPT, branch_A_np, branch_B_np, seed=sd,
                         guidance=GUIDANCE_SCALE, steps=NUM_INFERENCE_STEPS))
    im.save(OUTPUT_DIR / f"generated_seed{sd}.png")
    ax.imshow(im); ax.set_title(f"seed {sd}"); ax.axis("off")
plt.tight_layout()
plt.show()

## Notes

- **Input size:** the model was trained on 256 × 256 tiles. For larger areas, split your rasters into 256 × 256 tiles and generate each one.
- **Prompt:** use a prompt similar in style to the training prompts. The geospatial layers control the layout, while the prompt mainly influences style and appearance.
- **Guidance:** `GUIDANCE_SCALE = 1.0` is the setting we used for evaluation. Higher values add classifier-free guidance and may change colors and contrast.
- **Memory:** if the GPU runs out of memory, restart the runtime and run the notebook once. The model needs roughly 6–8 GB of VRAM.
- **Aligned inputs:** the six layers must cover exactly the same area and have the same resolution.